# Métricas 01 — Inventario: qué ofrece cada fuente

Antes de definir los dashboards y las métricas, este notebook muestra **qué se puede contar** con cada fuente. Es solo lectura: lee el diccionario y los CSV de Bronze y las tablas de Silver, y no escribe nada en las capas.

| Fuente | Qué aporta | Cómo se lee aquí |
| --- | --- | --- |
| **RENAMU** (INEI) | Cómo es la municipalidad y qué hace: personal, equipamiento, servicios | Diccionario PDF 2022 (referencia) + presencia de cada variable en las ediciones 2020–2026 |
| **SISMEPRE** (MEF) | Gestión del impuesto predial: emisión, cobranza, saldos; clasificación A–G | Catálogo de formularios y preguntas + columnas del ESAT |
| **SIAF** (MEF) | Cuánto dinero: presupuesto y recaudación por rubro y clasificador | Tabla Silver `siaf_ingreso` |

**Período del análisis:** 2020–2025/26. En el RENAMU, la edición *N* describe a la municipalidad al 31 de diciembre de *N − 1*; por ejemplo, la edición 2022 trae el personal al 31/12/2021.

Al final se propone la **historia** y los **6 dashboards**. Las métricas están en `metricas/metricas.md`.

## 0. Configuración

In [1]:
import sys, json, logging
from pathlib import Path
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ARQUITECTURA.md").exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
logging.getLogger("pdfminer").setLevel(logging.ERROR)

import pandas as pd
import duckdb
from src import bronze
from src.perfilado import detectar_formato
from src.renamu import estructura_diccionario

pd.set_option("display.max_columns", 30); pd.set_option("display.width", 230)
pd.set_option("display.max_colwidth", 80); pd.set_option("display.max_rows", 200)
man = pd.DataFrame(bronze.leer_manifest()["archivos"])
f3 = man[(man["fuente"] == "F3") & man["anio"].between(2020, 2026) & ~man["ruta"].str.contains("_F02_")]
EDICIONES = list(range(2020, 2027))

## 1. RENAMU

### 1.1 Secciones del diccionario 2022

Cada sección agrupa preguntas, y cada pregunta tiene varios campos (columnas del CSV). La columna `en_las_7_ediciones` dice qué parte de los campos de 2022 existe con el mismo código en **todas** las ediciones 2020–2026.

In [2]:
pdf22 = RAIZ / f3[(f3["anio"] == 2022) & f3["archivo"].str.endswith(".pdf")]["ruta"].iloc[0]
dic = estructura_diccionario([pdf22])
dic = dic[~dic["campo"].str.startswith("VFI")]          # la bandera "municipalidad informante" no es una variable

# columnas de cada edición (2020 viene en 19 módulos: se juntan)
cols = {a: set() for a in EDICIONES}
for r in f3[f3["archivo"].str.lower().str.endswith(".csv")].itertuples():
    cols[int(r.anio)] |= {c.upper() for c in detectar_formato(RAIZ / r.ruta)["columnas"]}
dic["ediciones"] = dic["campo"].map(lambda c: sum(c in cols[a] for a in EDICIONES))

secciones = (dic.groupby("seccion_canonica", sort=False)
               .agg(preguntas=("nro_pregunta", "nunique"), campos=("campo", "count"),
                    en_las_7_ediciones=("ediciones", lambda s: f"{100 * (s == 7).mean():.0f} %"))
               .rename_axis("seccion"))
display(secciones)
print(f"Total: {dic['nro_pregunta'].nunique()} preguntas · {len(dic)} campos · "
      f"{100 * (dic['ediciones'] == 7).mean():.0f} % presentes en las 7 ediciones")

,preguntas,campos,en_las_7_ediciones
seccion,,,
DATOS GENERALES DE LA MUNICIPALIDAD,7,19,100 %
EQUIPAMIENTO Y TECNOLOGÍAS DE LA INFORMACIÓN Y COMUNICACIONES,7,86,94 %
RECURSOS HUMANOS,4,117,100 %
PLANIFICACIÓN MUNICIPAL,8,81,94 %
GESTIÓN DE ADMINISTRACIÓN TRIBUTARIA,2,8,88 %
LICENCIAS OTORGADAS POR LA MUNICIPALIDAD,1,4,100 %
DESARROLLO URBANO Y/O RURAL,4,26,100 %
SANEAMIENTO AMBIENTAL Y SALUBRIDAD,9,50,92 %
"EDUCACIÓN, CULTURA, DEPORTE Y RECREACIÓN",9,68,29 %


Total: 85 preguntas · 724 campos · 88 % presentes en las 7 ediciones


**Lectura:** el RENAMU no es solo finanzas. Trae 4 módulos que describen a la municipalidad:

| Módulo | Qué cuenta | Ejemplos |
| --- | --- | --- |
| II Equipamiento y TIC | Con qué trabaja | Vehículos, computadoras, internet, portal de transparencia |
| III Recursos humanos | Con quién trabaja | Personal por régimen y sexo, funcionarios, personal con discapacidad, asistencia técnica que necesita |
| IV Competencias y funciones | Cómo se gestiona | Planificación, **administración tributaria** (personal de rentas, ejecución coactiva), licencias, desarrollo urbano |
| V Servicios públicos locales | Qué servicios presta | Residuos sólidos, seguridad ciudadana (serenazgo), programas sociales (Vaso de Leche), DEMUNA, cultura y deporte, riesgo de desastres, ambiente, **finanzas** |

El módulo I (datos generales) es el directorio: dirección, teléfonos, correo y alcalde. Son datos de contacto y no se usan.

### 1.2 Variables candidatas para métricas

Una selección de las variables con más potencial: son cantidades o indicadores de capacidad y servicio, con buen llenado. Para cada una: en cuántas ediciones existe, qué porcentaje de municipalidades la respondió en 2022 y el total nacional (o el porcentaje de *sí*, si es una pregunta de sí/no).

In [3]:
CANDIDATAS = [   # (tema, código, descripción, tipo)
    ("Capacidad: personal", "P19D_T", "Personal total", "cantidad"),
    ("Capacidad: personal", "P19_1_T", "Funcionarios y/o directivos", "cantidad"),
    ("Capacidad: personal", "P32", "¿Tiene personal exclusivo de administración tributaria?", "si_no"),
    ("Capacidad: personal", "P32_1_T", "Personal exclusivo de administración tributaria", "cantidad"),
    ("Capacidad: equipamiento", "P14A_1", "Computadoras con acceso a internet", "cantidad"),
    ("Capacidad: equipamiento", "P11A_1_1", "Autos y/o camionetas operativos", "cantidad"),
    ("Capacidad: gestión", "P31_3", "Acepta pago en línea (página web)", "marca"),
    ("Capacidad: gestión", "P96", "¿Tiene oficina o unidad ambiental?", "si_no"),
    ("Servicio: seguridad", "P69_1_T", "Efectivos de serenazgo", "cantidad"),
    ("Servicio: seguridad", "P70A_1_2", "Autos y/o camionetas operativos del serenazgo", "cantidad"),
    ("Servicio: seguridad", "P71A_1", "Intervenciones por robo a transeúntes", "cantidad"),
    ("Servicio: residuos", "P41_1", "Residuos sólidos recogidos por día (kg)", "cantidad"),
    ("Servicio: residuos", "P44_1", "¿Los residuos van a relleno sanitario?", "si_no"),
    ("Servicio: residuos", "P44_2_1", "% de residuos que va a botadero", "porcentaje"),
    ("Servicio: social", "P60A_1_2", "Beneficiarios del Vaso de Leche", "cantidad"),
    ("Servicio: social", "P60A_2_2", "Beneficiarios de comedores populares", "cantidad"),
    ("Servicio: social", "P64_1_1", "Casos de alimentos atendidos por la DEMUNA", "cantidad"),
    ("Servicio: urbano", "P35A_1", "Licencias de edificación de viviendas unifamiliares", "cantidad"),
    ("Servicio: cultura y deporte", "P51_2", "Visitantes de bibliotecas municipales", "cantidad"),
    ("Servicio: riesgo", "P84A_1A", "Brigadas de emergencia", "cantidad"),
    ("Finanzas", "C96", "Total de ingresos ejecutados", "soles"),
    ("Finanzas", "C97", "Total de gastos ejecutados", "soles"),
]
csv22 = RAIZ / f3[(f3["anio"] == 2022) & f3["archivo"].str.lower().str.endswith(".csv")]["ruta"].iloc[0]
d22 = pd.read_csv(csv22, sep=";", dtype=str, encoding="utf-8-sig")
d22.columns = [c.upper() for c in d22.columns]
filas = []
for tema, c, desc, tipo in CANDIDATAS:
    v = d22[c].fillna("").str.strip()
    n = pd.to_numeric(v.str.replace(",", "."), errors="coerce")
    valor = {"cantidad": f"{n.sum():,.0f}", "soles": f"S/ {n.sum() / 1e6:,.0f} mill.",
             "si_no": f"{100 * (v == '1').mean():.0f} % sí", "marca": f"{100 * (~v.isin(['', '0'])).mean():.0f} % sí",
             "porcentaje": f"promedio {n.mean():.0f} %"}[tipo]
    filas.append({"tema": tema, "codigo": c, "descripcion": desc,
                  "ediciones_2020_2026": f"{sum(c in cols[a] for a in EDICIONES)}/7",
                  "con_dato_2022": f"{100 * (v != '').mean():.0f} %", "valor_nacional_2022": valor})
display(pd.DataFrame(filas).set_index(["tema", "codigo"]))

descripcion ediciones_2020_2026 con_dato_2022 valor_nacional_2022
tema                        codigo                                                                                                                 
Capacidad: personal         P19D_T                                             Personal total                 7/7         100 %             177,050
                            P19_1_T                               Funcionarios y/o directivos                 7/7         100 %              11,714
                            P32       ¿Tiene personal exclusivo de administración tributaria?                 7/7         100 %             63 % sí
                            P32_1_T           Personal exclusivo de administración tributaria                 7/7          63 %               9,753
Capacidad: equipamiento     P14A_1                         Computadoras con acceso a internet                 7/7          96 %              90,556
                            P11A_1_1                          Autos y/o camionetas operativos                 7/7          81 %               6,320
Capacidad: gestión          P31_3                           Acepta pago en línea (página web)                 7/7         100 %              3 % sí
                            P96                            ¿Tiene oficina o unidad ambiental?                 7/7         100 %             72 % sí
Servicio: seguridad         P69_1_T                                    Efectivos de serenazgo                 7/7          64 %              34,318
                            P70A_1_2            Autos y/o camionetas operativos del serenazgo                 7/7          60 %               2,804
                            P71A_1                      Intervenciones por robo a transeúntes                 7/7          25 %              22,803
Servicio: residuos          P41_1                     Residuos sólidos recogidos por día (kg)                 7/7          99 %          25,665,148
                            P44_1                      ¿Los residuos van a relleno sanitario?                 7/7          99 %             17 % sí
                            P44_2_1                           % de residuos que va a botadero                 7/7          82 %       promedio 89 %
Servicio: social            P60A_1_2                          Beneficiarios del Vaso de Leche                 7/7         100 %           2,095,197
                            P60A_2_2                     Beneficiarios de comedores populares                 7/7          59 %             699,231
                            P64_1_1                Casos de alimentos atendidos por la DEMUNA                 7/7          70 %              59,646
Servicio: urbano            P35A_1        Licencias de edificación de viviendas unifamiliares                 7/7          47 %              54,036
Servicio: cultura y deporte P51_2                       Visitantes de bibliotecas municipales                 6/7          16 %             274,025
Servicio: riesgo            P84A_1A                                    Brigadas de emergencia                 6/7          43 %               1,690
Finanzas                    C96                                  Total de ingresos ejecutados                 7/7         100 %     S/ 40,944 mill.
                            C97                                    Total de gastos ejecutados                 7/7         100 %     S/ 32,300 mill.

**Lectura:**
- Las cantidades de personal, serenazgo, residuos y Vaso de Leche están **en casi todas las municipalidades** y sirven para comparar capacidad y servicios.
- Algunas variables solo las responde quien tiene el servicio. Por ejemplo, el personal de rentas existe donde hay personal exclusivo (63 %): una celda vacía significa "no tiene", no "no informó".
- Todas se refieren al **31 de diciembre del año anterior a la edición**, así que se cruzan con el SIAF del año del dato.
- **Codificación:** en las preguntas de sí/no, 1 = sí y 2 = no. En las de opción múltiple (por ejemplo, las modalidades de pago), cada columna guarda el **número de la opción** si se marcó (3 = pago por la web) y 0 si no.
- **Limitación:** el RENAMU no trae la población del distrito. Las métricas "por habitante" necesitarían otra fuente (INEI). Mientras tanto, se compara entre municipalidades del mismo tipo (provincial o distrital, A–G) o por trabajador.

## 2. SISMEPRE

### 2.1 Formularios de la meta del predial (2022–2026)

El SISMEPRE no tiene diccionario oficial, pero los títulos de los formularios y el texto de las preguntas dicen qué mide.

In [4]:
B = RAIZ / "data/bronze/sismepre_predial"
form = pd.read_csv(B / "rentas_formulario.csv", dtype=str)
form = form[form["ANO_APLICACION"].astype(int) >= 2022]
display(form.groupby("TITULO").agg(anios=("ANO_APLICACION", lambda s: ", ".join(sorted(set(s))))).sort_values("anios"))
preg = pd.read_csv(B / "rentas_preguntas.csv", dtype=str)
print("Preguntas por año:", preg.groupby("ANO_APLICACION").size().to_dict())

,anios
TITULO,
2E. MACROPROCESO DE TESORERÍA Y PAGO,"2022, 2023, 2024, 2025"
2A. ASPECTOS GENERALES,"2022, 2023, 2024, 2025, 2026"
2B. MACROPROCESO DE REGISTRO Y DETERMINACIÓN DE LA DEUDA,"2022, 2023, 2024, 2025, 2026"
2C. MACROPROCESO DE NOTIFICACIONES,"2022, 2023, 2024, 2025, 2026"
2D. MACROPROCESO DE GESTIÓN DE CUENTA,"2022, 2023, 2024, 2025, 2026"
3A. INFORMACIÓN SOBRE LA EMISIÓN INICIAL TOTAL DE IMPUESTO PREDIAL,"2022, 2023, 2024, 2025, 2026"
3B. INFORMACIÓN SOBRE LA RECAUDACIÓN DE IMPUESTO PREDIAL (INSOLUTO),"2022, 2023, 2025, 2026"
3C. INFORMACIÓN SOBRE SALDOS POR COBRAR DEL MONTO INSOLUTO DE IMPUESTO PREDIAL,"2022, 2023, 2025, 2026"
3B. INFORMACIÓN SOBRE LA RECAUDACIÓN DE IMPUESTO PREDIAL (INSOLUTO + REAJUSTE),"2023, 2024, 2025"


Preguntas por año: {'2020': 91, '2022': 91, '2023': 234, '2024': 126, '2025': 154, '2026': 140}


**Lectura:** los formularios cubren dos bloques:
- **2A–2E, gestión de la administración tributaria (ATM):** procesos de registro, notificación, cuenta corriente y cobranza. Son preguntas de sí/no y cantidades, por ejemplo valores notificados y fraccionamientos.
- **3A–3E y 4A–4E, cifras:** emisión inicial y final, recaudación y saldos por cobrar del impuesto predial y de los arbitrios, y contribuyentes con beneficio tributario. Estas cifras están en el **ESAT** (2.2).

### 2.2 Estadística del predial (ESAT)

Una fila por municipalidad, formulario y mes o año de la estadística. Se muestran las columnas de montos y cantidades con su total del año estadístico 2023.

In [5]:
esat = pd.read_csv(B / "rentas_esat_estadistica_atm.csv", dtype=str)
x = esat[esat["ANO_ESTADISTICA"] == "2023"]
QUE = {"EMISIONPREDIAL": "Emisión del predial (lo que se cobra)", "RECAUDACTUAL": "Recaudación del año",
       "RECAUDANTER": "Recaudación de deuda de años anteriores", "SALDOPREDIAL": "Saldo por cobrar (deuda)",
       "BASEIMPONIBLE": "Base imponible", "ADULTOMAYOR": "Beneficio adulto mayor", "PREDIO": "Predios", "CONTRI": "Contribuyentes"}
filas = []
for c in [c for c in esat.columns if c.startswith(("MON_", "NUM_"))]:
    n = pd.to_numeric(x[c], errors="coerce")
    filas.append({"columna": c, "mide": next((v for k, v in QUE.items() if k in c), "—"),
                  "suma_2023": f"{n.sum():,.0f}", "con_valor_distinto_de_0": f"{100 * (n.fillna(0) != 0).mean():.0f} %"})
display(pd.DataFrame(filas).set_index("columna"))
print(f"Municipalidades con ESAT 2023: {x['SEC_EJEC'].nunique()} · formularios: {sorted(x['FORMULARIO_ID'].unique(), key=int)}")

,mide,suma_2023,con_valor_distinto_de_0
columna,,,
MON_EMISIONPREDIAL_AFECTO,Emisión del predial (lo que se cobra),"5,668,075,595",6 %
MON_EMISIONPREDIAL_EXON,Emisión del predial (lo que se cobra),0,0 %
NUM_EMISIONPREDIAL_AFECTO,Emisión del predial (lo que se cobra),"19,537,034",8 %
NUM_EMISIONPREDIAL_EXON,Emisión del predial (lo que se cobra),0,0 %
NUM_EMISIONPREDIAL_CASA,Emisión del predial (lo que se cobra),"15,907,196",7 %
NUM_EMISIONPREDIAL_OTROS,Emisión del predial (lo que se cobra),"24,306,969",7 %
MON_BASEIMPONIBLE_AFECTO,Base imponible,0,0 %
MON_BASEIMPONIBLE_EXON,Base imponible,0,0 %
MON_AUTOAVALUO_INAFECTO,—,0,0 %


Municipalidades con ESAT 2023: 1039 · formularios: ['6', '7', '8', '9', '10', '11', '13', '14']


**Lectura:**
- El ESAT permite medir la **gestión** del predial, no solo cuánto entra: emisión (lo que debería cobrarse), recaudación del año y de años anteriores, cobranza **ordinaria vs. coactiva** y **saldo por cobrar** (la deuda acumulada).
- **Cuidado al sumar:** cada formulario (3A emisión inicial, 3B recaudación, 3C saldos, 3D emisión final, …) y cada mes es una fila distinta, así que las sumas de arriba mezclan formularios. En Silver (`04_sismepre`) cada medida debe tomarse **de su formulario y del cierre del año**.
- Varias columnas están en 0 en 2023 (base imponible, adulto mayor, predios); dependen del formulario y del año.
- **Clasificación A–G** (`rentas_entidad_estado.csv`): estable desde 2019. Sirve para comparar municipalidades parecidas.

## 3. SIAF

Ya está limpio en Silver (`siaf_ingreso`, 2009–2026). Para el análisis se usa 2020 en adelante.

In [6]:
S = f"read_parquet('{RAIZ}/data/silver/siaf_ingreso/*/*.parquet', hive_partitioning=true)"
con = duckdb.connect()
display(con.sql(f"""select rubro, any_value(rubro_nombre) rubro_nombre,
        {", ".join(f"round(sum(case when anio={a} then monto_recaudado end)/1e6) as \"{a}\"" for a in range(2020, 2026))}
        from {S} where anio between 2020 and 2025 group by 1 order by 1""").df().set_index("rubro"))
print("Recaudado por rubro, millones de soles (2020–2025). Ingresos propios: 08 y 09; transferencias: 07, 13 y 18; crédito: 19.")

,rubro_nombre,2020,2021,2022,2023,2024,2025
rubro,,,,,,,
07,FONDO DE COMPENSACION MUNICIPAL,6462.0,8778.0,10412.0,9504.0,9690.0,9520.0
08,IMPUESTOS MUNICIPALES,3212.0,3880.0,4261.0,4607.0,5294.0,5792.0
09,RECURSOS DIRECTAMENTE RECAUDADOS,3603.0,4282.0,4359.0,4514.0,5233.0,5828.0
13,DONACIONES Y TRANSFERENCIAS,1158.0,1871.0,2166.0,2752.0,3048.0,2533.0
18,"CANON Y SOBRECANON, REGALIAS, RENTA DE ADUANAS Y PARTICIPACIONES",9607.0,12804.0,19676.0,19028.0,18342.0,17730.0
19,RECURSOS POR OPERACIONES OFICIALES DE CREDITO,9418.0,9405.0,4832.0,2135.0,3327.0,4759.0


Recaudado por rubro, millones de soles (2020–2025). Ingresos propios: 08 y 09; transferencias: 07, 13 y 18; crédito: 19.


**Lectura:** el SIAF responde **cuánto** dinero hay, de **dónde** viene y **en qué** concepto entra (el clasificador, por ejemplo el predial). No dice nada de la capacidad ni de los servicios de la municipalidad: eso lo aportan el RENAMU y el SISMEPRE. Las tres fuentes se cruzan por **`ubigeo` + año del dato** (el SISMEPRE, por `sec_ejec`).

## 4. Propuesta: historia y 6 dashboards

**Historia central:** *¿Las municipalidades tienen la capacidad para generar y gestionar sus propios ingresos, y cómo se refleja eso en los servicios que prestan?*

| # | Dashboard | Pregunta que responde | Fuentes |
| --- | --- | --- | --- |
| 1 | **Panorama financiero** | ¿Cuánto dinero tienen, de dónde viene y cuánto ejecutan de lo presupuestado? | SIAF |
| 2 | **Autonomía vs. dependencia** | ¿Quién vive de sus propios ingresos y quién de transferencias (FONCOMUN, canon)? ¿Cuánto gasto cubren? | SIAF + RENAMU (finanzas) |
| 3 | **Capacidad institucional** | ¿Con qué personal y equipamiento cuentan? ¿Las que tienen más capacidad recaudan más? | RENAMU + SIAF |
| 4 | **Gestión del predial** | ¿Cuánto de lo que emiten cobran? ¿Cuánta deuda acumulan? ¿Influye tener personal de rentas o coactiva? | SISMEPRE + SIAF + RENAMU |
| 5 | **Servicios a la ciudadanía** | ¿Qué servicios prestan (seguridad, residuos, programas sociales) y cómo se relacionan con sus recursos? | RENAMU + SIAF |
| 6 | **Calidad de datos** | ¿Qué tan confiables y al día están los datos? ¿Qué se excluyó y por qué? | Bitácoras y auditoría |

Las métricas de cada dashboard, con su fórmula, fuente, grano y lectura, están en **`metricas/metricas.md`**. El siguiente paso es un prototipo visual de estos 6 dashboards con datos reales.